# API Exploration — vlrdevapi

Goal: understand the vlrdevapi endpoints and map VCT team IDs for the prediction pipeline.

## Setup

In [ ]:
import vlrdevapi as vlr
import pandas as pd

## 1. Search Endpoint
Testing team search to understand response format and extract team IDs.

In [ ]:
result = vlr.search.search("Sentinels")
team = result.teams[0]
print(team)

SENTINELS_ID = team.team_id

SearchTeamResult(team_id=2, url='https://www.vlr.gg/search/r/team/2/idx', name='Sentinels', country='United States', logo_url='https://owcdn.net/img/62875027c8e06.png', is_inactive=False, result_type='team')


### Findings
- Returns SearchTeamResult with team_id, name, country, is_inactive
- First result is usually the correct team
- team_id is what we need for placements()

## 2. Placements Endpoint
Testing placements to understand historical results format.

In [21]:
result = vlr.teams.placements(team_id=SENTINELS_ID)
for p in result[:3]:
    print(p)

EventPlacement(event_id=2682, event_name='VCT 2026: Americas Kickoff', event_url='https://www.vlr.gg/event/2682/vct-2026-americas-kickoff', placements=[PlacementDetail(series='Main Event', place='9th', prize_money=None)], year='2026')
EventPlacement(event_id=2748, event_name='Sentinels Invitational 2025', event_url='https://www.vlr.gg/event/2748/sentinels-invitational-2025', placements=[PlacementDetail(series='Main Event', place='1st', prize_money='$7,500')], year='2025')
EventPlacement(event_id=2602, event_name='Red Bull Home Ground 2025', event_url='https://www.vlr.gg/event/2602/red-bull-home-ground-2025', placements=[PlacementDetail(series='Main Event', place='5th', prize_money='$3,000')], year='2025')


In [5]:
result = vlr.teams.placements(team_id=2)
print(result)

[EventPlacement(event_id=2682, event_name='VCT 2026: Americas Kickoff', event_url='https://www.vlr.gg/event/2682/vct-2026-americas-kickoff', placements=[PlacementDetail(series='Main Event', place='9th', prize_money=None)], year='2026'), EventPlacement(event_id=2748, event_name='Sentinels Invitational 2025', event_url='https://www.vlr.gg/event/2748/sentinels-invitational-2025', placements=[PlacementDetail(series='Main Event', place='1st', prize_money='$7,500')], year='2025'), EventPlacement(event_id=2602, event_name='Red Bull Home Ground 2025', event_url='https://www.vlr.gg/event/2602/red-bull-home-ground-2025', placements=[PlacementDetail(series='Main Event', place='5th', prize_money='$3,000')], year='2025'), EventPlacement(event_id=2716, event_name='SEN City Classic 2025', event_url='https://www.vlr.gg/event/2716/sen-city-classic-2025', placements=[PlacementDetail(series='Main Event', place='1st', prize_money='$10,000')], year='2025'), EventPlacement(event_id=2283, event_name='Valoran

### Findings
- Returns list of EventPlacement with event_name, year, place
- Includes regional and international events mixed together
- place is a string ("1st", "5th") — needs parsing
- Includes All-Star/showmatch events — needs filtering

## 3. Team ID Mapping
Auto-mapping all VCT teams from matches.parquet to their VLR team IDs using the search endpoint.

> **Note:** `result.teams[0]` is not always correct — short/generic org names (T1, NRG, Gen.G) 
> can return academy teams first. We verify and correct these manually after mapping.

In [ ]:
# All showmatch teams, not competitive 
# i've checked them all before
INVALID_TEAMS = {
    "Team tarik", "Team Toast", "Team World", "Team Thailand",
    "Team International", "Precise Defeat", "Pure Aim", "Glory Once Again"
}

df = pd.read_parquet("../data/processed/matches.parquet")

all_teams = set(df["team1"].tolist() + df["team2"].tolist())
valid_teams = all_teams - INVALID_TEAMS
teams = sorted(valid_teams)

print(f"{len(teams)} valid teams\n")
print("\n".join(teams))

48 valid teams

100 Thieves
2Game Esports
All Gamers
Apeks
BBL Esports
BOOM Esports
Bilibili Gaming
Cloud9
DRX
DetonatioN FocusMe
Dragon Ranger Gaming
EDward Gaming
Evil Geniuses
FNATIC
FURIA
FUT Esports
FunPlus Phoenix
G2 Esports
GIANTX
Gen.G
Gentle Mates
Global Esports
JDG Esports
KOI
KRÜ Esports
Karmine Corp
LEVIATÁN
LOUD
MIBR
NRG
Natus Vincere
Nongshim RedForce
Nova Esports
Paper Rex
Rex Regum Qeon
Sentinels
T1
TALON
TYLOO
Team Heretics
Team Liquid
Team Secret
Team Vitality
Titan Esports Club
Trace Esports
Wolves Esports
Xi Lai Gaming
ZETA DIVISION


In [31]:
import time

team_id_map = {}
failed = []

for team_name in teams:
    try:
        result = vlr.search.search(team_name)

        if result.teams:
            team_id_map[team_name] = result.teams[0].team_id
        else:
            failed.append(team_name)
            
    except Exception:
        failed.append(team_name)
    time.sleep(0.5)  # avoid rate limiting

print(f"Mapped: {len(team_id_map)}")
print(f"Failed: {failed}\n")

# all teams mapped (i love pythons oneliners)
print("\n".join(f"{team}: {team_id}" for team, team_id in team_id_map.items()))

Mapped: 48
Failed: []

100 Thieves: 19510
2Game Esports: 15072
All Gamers: 1119
Apeks: 11479
BBL Esports: 397
BOOM Esports: 466
Bilibili Gaming: 12010
Cloud9: 188
DRX: 9749
DetonatioN FocusMe: 278
Dragon Ranger Gaming: 11981
EDward Gaming: 1120
Evil Geniuses: 9547
FNATIC: 2593
FURIA: 2406
FUT Esports: 20697
FunPlus Phoenix: 11328
G2 Esports: 257
GIANTX: 15119
Gen.G: 17392
Gentle Mates: 21093
Global Esports: 918
JDG Esports: 13576
KOI: 7035
KRÜ Esports: 2355
Karmine Corp: 8877
LEVIATÁN: 2359
LOUD: 6961
MIBR: 16606
NRG: 21049
Natus Vincere: 4915
Nongshim RedForce: 11060
Nova Esports: 12064
Paper Rex: 624
Rex Regum Qeon: 878
Sentinels: 2
T1: 15578
TALON: 8304
TYLOO: 731
Team Heretics: 1001
Team Liquid: 7055
Team Secret: 6199
Team Vitality: 2059
Titan Esports Club: 14137
Trace Esports: 12685
Wolves Esports: 13790
Xi Lai Gaming: 13581
ZETA DIVISION: 6997


### Verification
Short/generic names (T1, NRG, Gen.G, KOI) are ambiguous — inspecting all search 
results to find the correct IDs.

In [ ]:
# search all results for ambiguous names to find correct IDs
for name in ["T1", "NRG", "Gen.G", "KOI"]:
    result = vlr.search.search(name)

    print(f"\n{name}:")
    for team in result.teams:
        print(f"  id={team.team_id} | name={team.name} | inactive={team.is_inactive}")


T1:
  id=15578 | name=T1 Academy | inactive=False
  id=14 | name=T1 | inactive=False
  id=614 | name=T1 Korea(inactive since December 15, 2020) | inactive=True
  id=19775 | name=T1njA esports | inactive=False
  id=21749 | name=T1uA | inactive=False
  id=4075 | name=T1 Academy(inactive since February 8, 2022) | inactive=True
  id=20187 | name=T1 Spotlight | inactive=True
  id=17744 | name=T1reds Retirement Home | inactive=False
  id=2410 | name=TierTenTeam | inactive=True

NRG:
  id=21049 | name=NRG Academy | inactive=False
  id=1034 | name=NRG | inactive=False
  id=4411 | name=NRGeniuses | inactive=True

Gen.G:
  id=17392 | name=Gen.G Global Academy | inactive=False
  id=17 | name=Gen.G | inactive=False
  id=4475 | name=Gen.G Black(inactive since August 2, 2023) | inactive=True
  id=13403 | name=Gen.G Global Academy GC | inactive=True
  id=21858 | name=Gen.G GC | inactive=False

KOI:
  id=7035 | name=KOI | inactive=True
  id=17174 | name=KOI Fénix(inactive since September 28, 2025) | 

KOI returned `inactive=True` for all results — checking if they actually played in 2025.

In [ ]:

df[(df["team1"] == "KOI") | \
    (df["team2"] == "KOI")] \
    [["event", "date", "team1", "team2"]].head()

,event,date,team1,team2
8,VCT 2025 EMEA Kickoff_csvs,2025-01-16,Team Liquid,KOI
53,VCT 2025 EMEA Kickoff_csvs,2025-01-25,Gentle Mates,KOI
134,VCT 2025 EMEA Stage 1_csvs,2025-03-28,FUT Esports,KOI
160,VCT 2025 EMEA Stage 1_csvs,2025-04-05,KOI,Karmine Corp
177,VCT 2025 EMEA Stage 1_csvs,2025-04-09,KOI,BBL Esports


In [ ]:
# manual corrections — auto-mapping returned academy teams for these
team_id_map["T1"] = 14
team_id_map["NRG"] = 1034
team_id_map["Gen.G"] = 17

### Corrections

| Team | Auto ID | Issue | Correct ID |
|---|---|---|---|
| T1 | 15578 | T1 Academy returned first | 14 |
| NRG | 21049 | NRG Academy returned first | 1034 |
| Gen.G | 17392 | Gen.G Global Academy returned first | 17 |
| KOI | 7035 | inactive=True but played EMEA early 2025 | 7035 |

### Findings
- 48/48 teams mapped, 3 required manual correction
- KOI is inactive but ID is correct — disbanded mid-2025
- team_id_map is ready to move to constants.py

In [ ]:
# all ids corrected 
print("\n".join(f"{team}: {team_id}" for team, team_id in team_id_map.items()))

100 Thieves: 19510
2Game Esports: 15072
All Gamers: 1119
Apeks: 11479
BBL Esports: 397
BOOM Esports: 466
Bilibili Gaming: 12010
Cloud9: 188
DRX: 9749
DetonatioN FocusMe: 278
Dragon Ranger Gaming: 11981
EDward Gaming: 1120
Evil Geniuses: 9547
FNATIC: 2593
FURIA: 2406
FUT Esports: 20697
FunPlus Phoenix: 11328
G2 Esports: 257
GIANTX: 15119
Gen.G: 17
Gentle Mates: 21093
Global Esports: 918
JDG Esports: 13576
KOI: 7035
KRÜ Esports: 2355
Karmine Corp: 8877
LEVIATÁN: 2359
LOUD: 6961
MIBR: 16606
NRG: 1034
Natus Vincere: 4915
Nongshim RedForce: 11060
Nova Esports: 12064
Paper Rex: 624
Rex Regum Qeon: 878
Sentinels: 2
T1: 14
TALON: 8304
TYLOO: 731
Team Heretics: 1001
Team Liquid: 7055
Team Secret: 6199
Team Vitality: 2059
Titan Esports Club: 14137
Trace Esports: 12685
Wolves Esports: 13790
Xi Lai Gaming: 13581
ZETA DIVISION: 6997
